In [4]:
from types import SimpleNamespace
import numpy as np


class AM81111SystemPosition:

    driveUnit = SimpleNamespace(**{ 
        'gearBoxGearRatio': 1000.0,        
        'timingBeltTransmissionGearRatio': 2.0,
        'spindlePitch': 5.0,                        # mm
        'motorIncrementPositions': 262_144,         
        'cylinderDiameter': 15.0,                   # mm
        'limit': {
            'low': 0,
            'high': 24_185_993 
        }
    })

    @staticmethod
    def gearRatio():
        return AM81111SystemPosition.driveUnit.timingBeltTransmissionGearRatio * AM81111SystemPosition.driveUnit.gearBoxGearRatio

    @staticmethod
    def cylinderArea():
        """
        mm²
        """
        return np.pow(AM81111SystemPosition.driveUnit.cylinderDiameter,2) * np.pi / 4.
    
    @staticmethod
    def pitchVolume():
        """
        mm³
        """        
        return AM81111SystemPosition.driveUnit.spindlePitch * AM81111SystemPosition.cylinderArea()
    
    @staticmethod
    def turn2ml(value, bits):
        bitRange = 2 ** bits -1
        mtb = value[0] * AM81111SystemPosition.pitchVolume() / AM81111SystemPosition.gearRatio()
        stb = (value[1] / bitRange) * AM81111SystemPosition.pitchVolume() / AM81111SystemPosition.gearRatio() if bitRange > 0 else 0
        return (mtb + stb) / 1000 # mm³ -> ml

    @staticmethod
    def ml2turn(value, bits):
        bitRange = 2 ** bits -1
        mtb = round(np.trunc(value / AM81111SystemPosition.pitchVolume() * AM81111SystemPosition.gearRatio() * 1000), 0)
        stb = round(bitRange * (value - AM81111SystemPosition.turn2ml([mtb, 0], bits)) / AM81111SystemPosition.pitchVolume() * AM81111SystemPosition.gearRatio() * 1000, 0)
        return [np.uint32(mtb), np.uint32(stb)]


class AM81111ProfilePosition:

    """
    precision:  360°/2^bits
    cycles:     2^(32-bits)
    """
    @staticmethod
    def precision(bits, range=32):
        return 360 / (2**bits - 1) if bits > 0 else 360

    @staticmethod
    def cycles(bits, range=32):
        return 2**(range - bits - 1)

    @staticmethod
    def split(value, bits, range=32):            
        value = bin(value)[2:].zfill(range)
        return [
            int(value[:range-bits].zfill(range),2), 
            int(value[-bits:].zfill(range),2)
            ]
    
    @staticmethod
    def merge(value, bits, range=32, verbose=False):          
        rc = AM81111ProfilePosition.value(int("".join([
            bin(value[0])[2:].zfill(range-bits), 
            bin(value[1])[2:].zfill(bits)]), 2), range)
        return rc
    
    @staticmethod
    def value(value, range=32):
        rc = (2**range - 1) + value if value < 0 else value
        return rc
        
    @staticmethod
    def compare(raw, value, bits, range=32):
        raw = AM81111ProfilePosition.split(raw, bits, range)
        if raw[0] < value[0]:
            return -1
        elif raw[0] > value[0]:
            return +1
        else:
            if raw[1] < value[1]:
                return -1
            elif raw[1] > value[1]:
                return +1
        return 0        
    
    @staticmethod
    def sub(a, b, bits):
        return AM81111ProfilePosition.merge(a, bits) - AM81111ProfilePosition.merge(b, bits)

    @staticmethod
    def add(a, b, bits):
        return AM81111ProfilePosition.merge(a, bits) + AM81111ProfilePosition.merge(b, bits) 
    

In [5]:
bits = 6
value = 2 ** 26 # 4088

turn = AM81111ProfilePosition.split(value, bits)

#AM81111ProfilePosition.precision(bits), AM81111ProfilePosition.cycles(bits)
#turn = [2**(32-bits)-1, 0]
#value = AM81111ProfilePosition.merge(turn, bits)
#prec = AM81111ProfilePosition.precision(bits)
#vol = AM81111SystemPosition.pitchVolume()

#vol * prec/360, vol * 1/(2**bits-1), vol * (2**(32-bits)-1), 9 * vol / 1000

#[[bits, AM81111SystemPosition.turn2ml([0,1], bits) * 10**6, 360/(2**bits-1)] for bits in range(1,16,1)]

#turn = AM81111SystemPosition.ml2turn(1, bits)

#turn = [2263, 34]

val = AM81111ProfilePosition.merge(turn, bits)
turn, val, AM81111ProfilePosition.split(val, bits), AM81111SystemPosition.turn2ml(turn, bits)

([1048576, 0], 67108864, [1048576, 0], np.float64(463.2466863277365))

In [3]:
2**26

67108864